In [ ]:
import time
import geopandas as gpd
import rasterio
from rasterstats import zonal_stats
import pandas as pd
import sys

# -----------------------------
# Inputs (FULL FILE PATHS)
# -----------------------------
shapefile_path = r"C:\Users\KyleSteen.AzureAD\Documents\LCOE_Workspace\Alaska\Alaska_3_10.shp"

lcoe_raster = r"C:\Users\KyleSteen.AzureAD\Documents\LCOE_Workspace\Alaska\Alaska_LCOE.tif"

output_csv = r"C:\Users\KyleSteen.AzureAD\Documents\LCOE_Workspace\Alaska\Alaska_LCOE.csv"
log_file = r"C:\Users\KyleSteen.AzureAD\Documents\LCOE_Workspace\Alaska\Alaska_LCOE_log.txt"

# -----------------------------
# Logger (FLUSH ENABLED)
# -----------------------------
def log(msg):
    print(msg, flush=True)
    with open(log_file, "a") as f:
        f.write(f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] {msg}\n")

# -----------------------------
# Start Script
# -----------------------------
log("Starting Alaska LCOE CSV export script.")

# -----------------------------
# Load Shapefile
# -----------------------------
log("Loading shapefile...")
gdf = gpd.read_file(shapefile_path)
total_polygons = len(gdf)
log(f"Loaded {total_polygons} polygons.")

# -----------------------------
# Check CRS
# -----------------------------
log("Checking CRS information...")

vector_crs = gdf.crs
log(f"Shapefile CRS: {vector_crs}")

with rasterio.open(lcoe_raster) as src:
    raster_crs = src.crs

log(f"Raster CRS: {raster_crs}")

if vector_crs != raster_crs:
    log("WARNING: CRS mismatch detected!")
    log("Reproject shapefile to match raster before running for accurate results.")
else:
    log("CRS match confirmed.")

# -----------------------------
# Compute Zonal Statistics with Progress
# -----------------------------
log("Beginning zonal statistics computation...")

start_time = time.time()
lcoe_means = []

for idx, row in gdf.iterrows():
    
    stat = zonal_stats(
        row.geometry,
        lcoe_raster,
        stats=["mean"],
        all_touched=True,
        nodata=-9999
    )[0]["mean"]
    
    lcoe_means.append(stat)

    # ---- Progress Update ----
    percent_complete = ((idx + 1) / total_polygons) * 100

    if (idx + 1) % max(1, total_polygons // 100) == 0 or (idx + 1) == total_polygons:
        elapsed = time.time() - start_time
        log(f"Progress: {percent_complete:.1f}% | "
            f"{idx+1}/{total_polygons} polygons | "
            f"Elapsed: {elapsed/60:.2f} minutes")

# -----------------------------
# Build Output DataFrame
# -----------------------------
log("Building output dataframe...")

output_df = pd.DataFrame({
    "ROW_ID": gdf["ROW_ID"],
    "LCOE_Mean": lcoe_means
})

# -----------------------------
# Export CSV
# -----------------------------
output_df.to_csv(output_csv, index=False)
log(f"CSV successfully saved: {output_csv}")

total_time = (time.time() - start_time) / 60
log(f"Script completed successfully in {total_time:.2f} minutes.")